# Demo: Generate Mask Images with `generate_mask`
This notebook demonstrates how to use the `generate_mask` function from `ACE/core/utils.py` to generate and visualize mask images. All relevant function definitions are included below for a self-contained demo.

In [ ]:
# --- Imports and device setup ---
import torch
import numpy as np
from PIL import Image
import cv2
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Helper: restore_img ---
def restore_img(img, x, norm_fn=lambda x: (x - 0.5) / 0.5):
    a,b,c = img.shape
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.reshape((1, a, b, c))
    img = img.astype('float32') / 255.0
    img = np.transpose(img, axes=(0, 3, 1, 2))
    img = norm_fn(img)
    return img

# --- Helper: presence_of_smile (uses dlib) ---
def presence_of_smile(image, image2, idx, batch_size):
    import dlib
    detector = dlib.get_frontal_face_detector()
    predictor = dlib.shape_predictor('/home/tpei0009/shape_predictor_68_face_landmarks.dat')
    image = np.array(image)
    image2 = np.array(image2)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)
    if not faces:
        print('Debug: Nofaces.')
        img_npy = restore_img(image, batch_size)
        img_npy2 = restore_img(image2, batch_size)
        return image, img_npy, img_npy2
    face = faces[0]
    landmarks = predictor(gray, face)
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    landmarks_coords = np.array([(landmarks.part(i).x, landmarks.part(i).y) for i in range(68)])
    hull = cv2.convexHull(landmarks_coords)
    cv2.fillConvexPoly(mask, hull, 255)
    image = cv2.bitwise_and(image, image, mask=mask)
    image2 = cv2.bitwise_and(image2, image2, mask=mask)
    img_npy = restore_img(image, batch_size)
    img_npy2 = restore_img(image2, batch_size)
    return image, img_npy, img_npy2

# --- Helper: confounder_mask ---
def confounder_mask(img, img2, batch_size, denorm_fn=lambda x: x * 0.5 + 0.5):
    img = denorm_fn(img)
    img2 = denorm_fn(img2)
    img = np.transpose(img, axes=(0, 2, 3, 1))
    img2 = np.transpose(img2, axes=(0, 2, 3, 1))
    img = (img * 255).astype('uint8')
    img2 = (img2 * 255).astype('uint8')
    output_images = []
    output_images2 = []
    for idx, i in enumerate(img):
        i = Image.fromarray(i)
        img2_ = Image.fromarray(img2[idx])
        image, img_npy, img_npy2 = presence_of_smile(i, img2_, idx, batch_size)
        output_images.append(img_npy)
        output_images2.append(img_npy2)
    stacked_img_npy = np.vstack(output_images)
    stacked_img_npy2 = np.vstack(output_images2)
    return stacked_img_npy, stacked_img_npy2

# --- Main: generate_mask ---
@torch.no_grad()
def generate_mask(x1, x2, dilation):
    x1_np = x1.detach().cpu().numpy()
    x2_np = x2.detach().cpu().numpy()
    batch_size = x1_np.shape[0]
    x1_mask, x2_mask = confounder_mask(x1_np, x2_np, batch_size)
    x1_mask = torch.from_numpy(x1_mask).to(device)
    x2_mask = torch.from_numpy(x2_mask).to(device)
    assert (dilation % 2) == 1, 'dilation must be an odd number'
    mask = (x1_mask - x2_mask).abs().sum(dim=1, keepdim=True)
    mask = mask / mask.view(mask.size(0), -1).max(dim=1)[0].view(-1, 1, 1, 1)
    mask = torch.where(mask > 0.35, mask, torch.tensor(0.0))
    dil_mask = F.max_pool2d(mask, dilation, stride=1, padding=(dilation - 1) // 2)
    return mask, dil_mask


In [ ]:
# --- Demo code (unchanged) ---
def load_image_as_tensor(path, size=(224, 224)):
    img = Image.open(path).convert('RGB').resize(size)
    img_np = np.array(img).astype(np.float32) / 255.0
    img_np = np.transpose(img_np, (2, 0, 1))  # (C, H, W)
    return torch.tensor(img_np).unsqueeze(0)  # (1, C, H, W)

img1_path = 'example1.jpg'
img2_path = 'example2.jpg'
x1 = load_image_as_tensor(img1_path)
x2 = load_image_as_tensor(img2_path)
x1 = x1.to(device)
x2 = x2.to(device)
mask, dil_mask = generate_mask(x1, x2, dilation=11)
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(np.transpose(x1[0].cpu().numpy(), (1, 2, 0)))
plt.title('Image 1')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(np.transpose(x2[0].cpu().numpy(), (1, 2, 0)))
plt.title('Image 2')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(mask[0, 0].cpu().numpy(), cmap='jet')
plt.title('Generated Mask')
plt.axis('off')
plt.show()
